# Actividad 2: Construcción de Muestra Representativa con PySpark
## Dataset: CIC-IDS2017 — Canadian Institute for Cybersecurity

**TC4034.10 Análisis de Grandes Volúmenes de Datos | Grupo 10 | Equipo #5**

- Luz Copelia Minutti Pérez — A01796921
- Karen Cecilia Varela Castro — A00958670
- Kevin Rosario Cota Rodríguez — A01796705
- Carlos Aaron Bocanegra Buitrón — A01796345

**Fecha:** 10 de Mayo de 2026

---

## Descripción general

En este cuaderno implementamos el proceso de particionamiento y muestreo representativo sobre el dataset **CIC-IDS2017**, desarrollado por el Canadian Institute for Cybersecurity de la University of New Brunswick. El dataset contiene flujos de tráfico de red capturados durante cinco días hábiles (lunes a viernes), con tráfico benigno y 14 tipos de ataques cibernéticos etiquetados. Cada registro representa un flujo de red descrito mediante 78 características estadísticas extraídas con la herramienta CICFlowMeter, más la columna `Label`.

El objetivo de este cuaderno es:
1. Cargar y limpiar el dataset en PySpark
2. Derivar las variables de caracterización (`attack_group`, `day_name`)
3. Aplicar las reglas de particionamiento (R1–R13)
4. Extraer sub-muestras de verificación por partición

**Enlace al dataset en Google Drive:** https://drive.google.com/drive/folders/13gL7O-5Mrnkx1l_1pixv0hFmuYoHbQOu

---
## Sección 1: Instalación de PySpark

Dado que ejecutamos este cuaderno en **Google Colab**, el entorno no incluye PySpark de manera predeterminada. Por ello, comenzamos instalándolo con `pip`. La bandera `-q` suprime la salida detallada de la instalación para mantener el cuaderno limpio. Esta celda debe ejecutarse al inicio de cada sesión de Colab, ya que el entorno se reinicia cada vez que se cierra la sesión.

In [ ]:
!pip install pyspark -q

---
## Sección 2: Montaje de Google Drive

Para acceder a los archivos CSV del dataset almacenados en nuestra cuenta de Google Drive, montamos la unidad dentro del entorno de Colab. Esto nos permite leer los datos directamente desde Drive sin necesidad de descargarlos al entorno local de Colab, lo que simplifica el flujo de trabajo y evita la pérdida de archivos al reiniciar la sesión. Al ejecutar esta celda, Colab solicita autorización de acceso a Google Drive a través de un enlace de autenticación.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## Sección 3: Inicialización de la Sesión de Spark

Iniciamos la sesión de Apache Spark mediante `SparkSession`, que es el punto de entrada para todas las operaciones de PySpark. Configuramos 4 GB de memoria para el driver (`spark.driver.memory = 4g`) con el fin de manejar el volumen del dataset (~840 MB en disco, expandidos en memoria al aplicar `inferSchema`). También importamos el módulo `functions as F`, que contiene las funciones de transformación que usaremos a lo largo del cuaderno (como `when`, `col`, `regexp_replace`, `input_file_name`), y `DoubleType` para el manejo de tipos numéricos de doble precisión.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = SparkSession.builder \
    .appName('CICIDS2017_Muestreo') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print(f'Spark versión: {spark.version}')
print('Sesión iniciada correctamente.')

Spark versión: 4.0.2
Sesión iniciada correctamente.


---
## Sección 4: Carga del Dataset

Cargamos los 8 archivos CSV del dataset CIC-IDS2017 de manera simultánea apuntando a la carpeta completa con el comodín `*.csv`. Usamos las opciones `header=true` para que PySpark reconozca la primera fila de cada archivo como encabezado, e `inferSchema=true` para que infiera automáticamente el tipo de dato de cada columna (entero, decimal, cadena, etc.) sin que tengamos que definirlos manualmente.

Después aplicamos `toDF()` con `c.strip()` sobre cada nombre de columna para eliminar los espacios en blanco al inicio y al final que el dataset original incluye por un problema de formato en los encabezados CSV. Sin este paso, columnas como `' Flow Duration'` (con espacio inicial) no serían reconocidas por sus nombres en las transformaciones posteriores.

In [ ]:
DATA_PATH = '/content/drive/MyDrive/BigData_PySpark/MachineLearningCVE/*.csv'

df_raw = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv(DATA_PATH)

# Quitar espacios en nombres de columnas
df_raw = df_raw.toDF(*[c.strip() for c in df_raw.columns])

print(f'Registros cargados: {df_raw.count():,}')
print(f'Columnas:           {len(df_raw.columns)}')

Registros cargados: 2,830,743
Columnas:           79


---
## Sección 5: Verificación Inicial de Etiquetas

Antes de realizar cualquier transformación, verificamos la distribución de la columna `Label` para confirmar que el dataset se cargó correctamente y entender la composición de la población objetivo. Agrupamos por `Label`, contamos los registros de cada categoría y ordenamos de mayor a menor frecuencia. Esta verificación nos permite identificar el fuerte desbalanceo de clases del dataset: el tráfico benigno (`BENIGN`) representa aproximadamente el 80% de los registros, mientras que los ataques son minoritarios. También nos permite detectar los caracteres corruptos (representados como `?`) presentes en las etiquetas de `Web Attack`, que corregiremos en la siguiente sección.

In [ ]:
df_raw.groupBy('Label') \
      .count() \
      .orderBy('count', ascending=False) \
      .show(20, truncate=False)

+--------------------------+-------+
|Label                     |count  |
+--------------------------+-------+
|BENIGN                    |2273097|
|DoS Hulk                  |231073 |
|PortScan                  |158930 |
|DDoS                      |128027 |
|DoS GoldenEye             |10293  |
|FTP-Patator               |7938   |
|SSH-Patator               |5897   |
|DoS slowloris             |5796   |
|DoS Slowhttptest          |5499   |
|Bot                       |1966   |
|Web Attack � Brute Force  |1507   |
|Web Attack � XSS          |652    |
|Infiltration              |36     |
|Web Attack � Sql Injection|21     |
|Heartbleed                |11     |
+--------------------------+-------+



---
## Sección 6: Limpieza de Datos

En esta sección aplicamos cuatro operaciones de limpieza que identificamos durante el análisis exploratorio (Actividad 1):

1. **Corrección de caracteres corruptos en `Label`:** Usamos `regexp_replace` con la expresión regular `[^\x20-\x7E]` para eliminar cualquier carácter fuera del rango ASCII imprimible. Esto corrige el problema de codificación UTF-8 en las etiquetas `Web Attack ? Brute Force`, `Web Attack ? XSS` y `Web Attack ? Sql Injection`, donde el separador original fue reemplazado por caracteres inválidos durante la generación del dataset.

2. **Eliminación de duraciones negativas:** Filtramos los registros con `Flow Duration < 0`, ya que una duración de tiempo negativa es físicamente imposible y representa un error de calidad del dataset (se detectó un valor de -13 µs en el EDA previo).

3. **Eliminación de nulos en `Flow Bytes/s`:** Filtramos los 1,358 registros con valor nulo en esta columna. Estos nulos se producen cuando `Flow Duration = 0`, lo que genera una división por cero al calcular bytes por segundo.

4. **Conversión de infinitos en `Flow Packets/s`:** Reemplazamos los valores infinitos (`Inf`) por `null` mediante un `when` condicional. PySpark no puede calcular estadísticas sobre valores infinitos, y estos se originan también por divisiones entre flujos de duración cero.

In [ ]:
# 1. Corregir caracteres corruptos en Label
df = df_raw.withColumn(
    'Label',
    F.regexp_replace(F.col('Label'), '[^\x20-\x7E]', '')
)

# 2. Eliminar duraciones negativas
df = df.filter(F.col('Flow Duration') >= 0)

# 3. Eliminar nulos en Flow Bytes/s
df = df.filter(F.col('Flow Bytes/s').isNotNull())

# 4. Convertir infinitos en Flow Packets/s a null
df = df.withColumn(
    'Flow Packets/s',
    F.when(F.col('Flow Packets/s') == float('inf'), None)
     .otherwise(F.col('Flow Packets/s').cast(DoubleType()))
)

print(f'Registros después de limpieza: {df.count():,}')

Registros después de limpieza: 2,830,628


---
## Sección 7: Verificación Post-Limpieza

Repetimos la distribución de etiquetas sobre el DataFrame limpio `df` para confirmar dos cosas: que los caracteres corruptos fueron eliminados correctamente de las etiquetas `Web Attack` (ahora deben aparecer sin el símbolo `?`), y que el número total de registros tras la limpieza es consistente con lo esperado. La reducción respecto al total original es mínima y corresponde a los registros con valores inválidos eliminados en la sección anterior.

In [ ]:
df.groupBy('Label') \
  .count() \
  .orderBy('count', ascending=False) \
  .show(20, truncate=False)

+-------------------------+-------+
|Label                    |count  |
+-------------------------+-------+
|BENIGN                   |2272982|
|DoS Hulk                 |231073 |
|PortScan                 |158930 |
|DDoS                     |128027 |
|DoS GoldenEye            |10293  |
|FTP-Patator              |7938   |
|SSH-Patator              |5897   |
|DoS slowloris            |5796   |
|DoS Slowhttptest         |5499   |
|Bot                      |1966   |
|Web Attack  Brute Force  |1507   |
|Web Attack  XSS          |652    |
|Infiltration             |36     |
|Web Attack  Sql Injection|21     |
|Heartbleed               |11     |
+-------------------------+-------+



---
## Sección 8: Creación de la Variable `attack_group`

Derivamos la primera variable de caracterización para el particionamiento: `attack_group`. Esta variable agrupa las 14 etiquetas originales de `Label` en 9 categorías conceptuales más manejables, usando `withColumn` combinado con una cadena de condiciones `when`/`otherwise`.

La razón de esta agrupación es evitar que el particionamiento genere demasiadas combinaciones con muy pocos registros: por ejemplo, `FTP-Patator` y `SSH-Patator` son ambos ataques de fuerza bruta con patrones de comportamiento similares, por lo que los agrupamos en `BruteForce`. De la misma manera, los cuatro tipos de ataques DoS (`Hulk`, `GoldenEye`, `Slowloris`, `Slowhttptest`) se consolidan en `DoS`. Esto reduce la cardinalidad de 14 a 9 clases activas, manteniendo el poder discriminativo necesario para el análisis.

Al final verificamos la distribución del `attack_group` resultante para confirmar que todas las etiquetas originales fueron asignadas correctamente.

In [ ]:
df = df.withColumn('attack_group',
    F.when(F.col('Label') == 'BENIGN', 'BENIGN')
     .when(F.col('Label').isin('FTP-Patator','SSH-Patator'), 'BruteForce')
     .when(F.col('Label').isin(
         'DoS Hulk','DoS GoldenEye',
         'DoS slowloris','DoS Slowhttptest'), 'DoS')
     .when(F.col('Label') == 'DDoS', 'DDoS')
     .when(F.col('Label').isin(
         'Web Attack  Brute Force',
         'Web Attack  XSS',
         'Web Attack  Sql Injection'), 'WebAttack')
     .when(F.col('Label') == 'Bot', 'Botnet')
     .when(F.col('Label') == 'PortScan', 'PortScan')
     .when(F.col('Label') == 'Infiltration', 'Infiltration')
     .when(F.col('Label') == 'Heartbleed', 'Heartbleed')
     .otherwise('Other')
)

# Verificar
df.groupBy('attack_group') \
  .count() \
  .orderBy('count', ascending=False) \
  .show()

+------------+-------+
|attack_group|  count|
+------------+-------+
|      BENIGN|2272982|
|         DoS| 252661|
|    PortScan| 158930|
|        DDoS| 128027|
|  BruteForce|  13835|
|   WebAttack|   2180|
|      Botnet|   1966|
|Infiltration|     36|
|  Heartbleed|     11|
+------------+-------+



---
## Sección 9: Exploración de Columnas Disponibles

Antes de derivar la variable `day_name` a partir del protocolo de red, exploramos los nombres exactos de todas las columnas disponibles en el DataFrame. Esta exploración fue necesaria porque al intentar crear la variable `protocol_name` a partir de la columna `Protocol`, obtuvimos un error que indicaba que dicha columna no existía en esta versión del dataset. Esta versión orientada a machine learning del CIC-IDS2017 no incluye la columna de protocolo, por lo que ajustamos nuestra estrategia de particionamiento para utilizar únicamente `attack_group` y `day_name` como variables de caracterización.

In [ ]:
# Ver todos los nombres de columnas
for col in df.columns:
    print(col)

Destination Port
Flow Duration
Total Fwd Packets
Total Backward Packets
Total Length of Fwd Packets
Total Length of Bwd Packets
Fwd Packet Length Max
Fwd Packet Length Min
Fwd Packet Length Mean
Fwd Packet Length Std
Bwd Packet Length Max
Bwd Packet Length Min
Bwd Packet Length Mean
Bwd Packet Length Std
Flow Bytes/s
Flow Packets/s
Flow IAT Mean
Flow IAT Std
Flow IAT Max
Flow IAT Min
Fwd IAT Total
Fwd IAT Mean
Fwd IAT Std
Fwd IAT Max
Fwd IAT Min
Bwd IAT Total
Bwd IAT Mean
Bwd IAT Std
Bwd IAT Max
Bwd IAT Min
Fwd PSH Flags
Bwd PSH Flags
Fwd URG Flags
Bwd URG Flags
Fwd Header Length34
Bwd Header Length
Fwd Packets/s
Bwd Packets/s
Min Packet Length
Max Packet Length
Packet Length Mean
Packet Length Std
Packet Length Variance
FIN Flag Count
SYN Flag Count
RST Flag Count
PSH Flag Count
ACK Flag Count
URG Flag Count
CWE Flag Count
ECE Flag Count
Down/Up Ratio
Average Packet Size
Avg Fwd Segment Size
Avg Bwd Segment Size
Fwd Header Length55
Fwd Avg Bytes/Bulk
Fwd Avg Packets/Bulk
Fwd Avg Bulk 

---
## Sección 10: Creación de la Variable `day_name`

Derivamos la segunda variable de caracterización: `day_name`, que captura el día de la semana en que fue capturado cada flujo de red. Para ello utilizamos la función `input_file_name()` de PySpark, que retorna la ruta completa del archivo CSV de origen de cada registro. Dado que los archivos del dataset están nombrados con el día de la semana (por ejemplo, `Monday-WorkingHours.pcap_ISCX.csv`), podemos extraer el día mediante `contains()` aplicado sobre el nombre del archivo.

Esta variable es fundamental para el particionamiento porque el comportamiento del dataset varía significativamente por día: el lunes contiene exclusivamente tráfico benigno, mientras que los distintos ataques fueron ejecutados de forma controlada entre martes y viernes, según el cronograma oficial de CIC-IDS2017. Sin esta dimensión temporal, la muestra perdería representatividad cronológica.

Verificamos la distribución resultante para confirmar que los 5 días están presentes y que los conteos son consistentes con el tamaño de los archivos CSV originales.

In [ ]:
df = df.withColumn('day_name',
    F.when(F.input_file_name().contains('Monday'),    'Monday')
     .when(F.input_file_name().contains('Tuesday'),   'Tuesday')
     .when(F.input_file_name().contains('Wednesday'), 'Wednesday')
     .when(F.input_file_name().contains('Thursday'),  'Thursday')
     .when(F.input_file_name().contains('Friday'),    'Friday')
     .otherwise('Unknown')
)

df.groupBy('day_name').count().orderBy('count', ascending=False).show()

+---------+------+
| day_name| count|
+---------+------+
|   Friday|703198|
|Wednesday|692682|
|   Monday|529903|
| Thursday|458953|
|  Tuesday|445892|
+---------+------+



---
## Sección 11: Verificación Cruzada día × tipo de ataque

Realizamos una verificación cruzada agrupando simultáneamente por `day_name` y `attack_group` para confirmar la coherencia lógica del particionamiento. Esta verificación es crítica porque `attack_group` y `day_name` no son variables independientes en CIC-IDS2017: el cronograma de captura determina qué tipos de ataque aparecen en cada día. Si la asignación fuera incorrecta, obtendríamos combinaciones imposibles como ataques DDoS el lunes o tráfico de Infiltración el martes.

El resultado esperado, y confirmado, es que el lunes contiene únicamente BENIGN; el martes tiene BENIGN y BruteForce; el miércoles tiene BENIGN, DoS y Heartbleed; el jueves tiene BENIGN, WebAttack e Infiltration; y el viernes tiene BENIGN, DDoS, PortScan y Botnet. Este comportamiento es consistente con la documentación oficial del dataset.

In [ ]:
df.groupBy('day_name', 'attack_group') \
  .count() \
  .orderBy('day_name', 'count', ascending=False) \
  .show(50, truncate=False)

+---------+------------+------+
|day_name |attack_group|count |
+---------+------------+------+
|Wednesday|BENIGN      |440010|
|Wednesday|DoS         |252661|
|Wednesday|Heartbleed  |11    |
|Tuesday  |BENIGN      |432057|
|Tuesday  |BruteForce  |13835 |
|Thursday |BENIGN      |456737|
|Thursday |WebAttack   |2180  |
|Thursday |Infiltration|36    |
|Monday   |BENIGN      |529903|
|Friday   |BENIGN      |414275|
|Friday   |PortScan    |158930|
|Friday   |DDoS        |128027|
|Friday   |Botnet      |1966  |
+---------+------------+------+



---
## Sección 12: Implementación del Particionamiento — Reglas R1 a R9

Implementamos la función `get_partition()`, que aplica las reglas de particionamiento mediante filtros sobre las dos variables de caracterización (`attack_group` y `day_name`). Cada llamada a esta función genera un DataFrame independiente que contiene únicamente los registros que cumplen simultáneamente ambas condiciones de la regla.

En esta celda generamos las primeras nueve particiones, correspondientes a los grupos de ataque y a la partición de tráfico benigno del lunes (R1). Las particiones R2 a R9 cubren todos los tipos de ataque identificados en el dataset, distribuidos en los días correspondientes según el cronograma de captura. PySpark aplica estos filtros de forma lazy (diferida), es decir, el procesamiento real sobre los datos no ocurre hasta que se invoca una acción como `count()` o `show()` en celdas posteriores.

In [ ]:
def get_partition(df, attack_grp, day):
    return df.filter(
        (F.col('attack_group') == attack_grp) &
        (F.col('day_name')     == day)
    )

df_R1 = get_partition(df, 'BENIGN',      'Monday')
df_R2 = get_partition(df, 'BruteForce',  'Tuesday')
df_R3 = get_partition(df, 'DoS',         'Wednesday')
df_R4 = get_partition(df, 'Heartbleed',  'Wednesday')
df_R5 = get_partition(df, 'WebAttack',   'Thursday')
df_R6 = get_partition(df, 'Infiltration','Thursday')
df_R7 = get_partition(df, 'Botnet',      'Friday')
df_R8 = get_partition(df, 'DDoS',        'Friday')
df_R9 = get_partition(df, 'PortScan',    'Friday')

---
## Sección 13: Cálculo de Probabilidades Empíricas — Primera Tabla (R1–R9)

Calculamos el número de registros de cada partición y su probabilidad de ocurrencia empírica, definida como P(R) = N_R / N_D, donde N_R es el número de registros de la partición y N_D es el total de registros del dataset tras la limpieza. Esta fórmula evita asumir independencia entre las variables de caracterización, lo cual sería incorrecto dado que `attack_group` y `day_name` están correlacionados por diseño del dataset.

Al presentar la primera versión de la tabla, identificamos que la suma de probabilidades era solo 0.3842, lo que indica que el 62% de los registros no estaba siendo capturado. Esto se debe a que el tráfico benigno de martes a viernes no estaba incluido en ninguna partición, solo el del lunes (R1). Esta observación nos llevó a agregar las cuatro particiones BENIGN faltantes en la siguiente sección.

In [ ]:
total = df.count()

particiones = {
    'R1 BENIGN/Monday':        df_R1,
    'R2 BruteForce/Tuesday':   df_R2,
    'R3 DoS/Wednesday':        df_R3,
    'R4 Heartbleed/Wednesday': df_R4,
    'R5 WebAttack/Thursday':   df_R5,
    'R6 Infiltration/Thursday':df_R6,
    'R7 Botnet/Friday':        df_R7,
    'R8 DDoS/Friday':          df_R8,
    'R9 PortScan/Friday':      df_R9,
}

print(f'{"Partición":<30} {"N registros":>12} {"P(partición)":>13}')
print('-' * 57)
suma = 0
for nombre, part in particiones.items():
    n = part.count()
    suma += n
    print(f'{nombre:<30} {n:>12,} {n/total:>13.4f}')
print('-' * 57)
print(f'{"TOTAL":<30} {suma:>12,} {suma/total:>13.4f}')

Partición                       N registros  P(partición)
---------------------------------------------------------
R1 BENIGN/Monday                    529,903        0.1872
R2 BruteForce/Tuesday                13,835        0.0049
R3 DoS/Wednesday                    252,661        0.0893
R4 Heartbleed/Wednesday                  11        0.0000
R5 WebAttack/Thursday                 2,180        0.0008
R6 Infiltration/Thursday                 36        0.0000
R7 Botnet/Friday                      1,966        0.0007
R8 DDoS/Friday                      128,027        0.0452
R9 PortScan/Friday                  158,930        0.0561
---------------------------------------------------------
TOTAL                             1,087,549        0.3842


---
## Sección 14: Identificación del Tráfico BENIGN Faltante

Al obtener una suma de probabilidades de solo 0.3842 en la tabla anterior, identificamos que el tráfico benigno de martes a viernes no estaba siendo capturado por ninguna de las particiones R1–R9. Para confirmar esto, filtramos la verificación cruzada mostrando únicamente los registros con `attack_group == 'BENIGN'` agrupados por día. Esta exploración revela que existen cuatro conjuntos adicionales de tráfico benigno (martes, miércoles, jueves y viernes) que deben tener su propia partición para que el esquema sea colectivamente exhaustivo y la suma de probabilidades sea igual a 1.0000.

In [ ]:
# Ver qué quedó fuera
df.groupBy('day_name', 'attack_group') \
  .count() \
  .filter(F.col('attack_group') == 'BENIGN') \
  .orderBy('day_name') \
  .show()

+---------+------------+------+
| day_name|attack_group| count|
+---------+------------+------+
|   Friday|      BENIGN|414275|
|   Monday|      BENIGN|529903|
| Thursday|      BENIGN|456737|
|  Tuesday|      BENIGN|432057|
|Wednesday|      BENIGN|440010|
+---------+------------+------+



---
## Sección 15: Completar el Particionamiento — Reglas R10 a R13

Añadimos las cuatro particiones de tráfico benigno faltantes (R10–R13), correspondientes al tráfico BENIGN de martes a viernes. Aunque estos días también contienen ataques, el tráfico benigno coexiste con ellos y representa un componente importante de la población: en el miércoles hay 440,010 registros BENIGN junto con los ataques DoS, y en el viernes hay 414,275 registros BENIGN junto con DDoS, PortScan y Botnet.

Agregamos estas cuatro particiones al diccionario existente y recalculamos la tabla completa. Ahora la suma de todos los registros debe ser igual al total post-limpieza (2,830,628) y la suma de probabilidades debe ser exactamente 1.0000, verificando que el particionamiento es mutuamente excluyente y colectivamente exhaustivo.

In [ ]:
df_R10 = get_partition(df, 'BENIGN', 'Tuesday')
df_R11 = get_partition(df, 'BENIGN', 'Wednesday')
df_R12 = get_partition(df, 'BENIGN', 'Thursday')
df_R13 = get_partition(df, 'BENIGN', 'Friday')

# Agregar al diccionario
particiones['R10 BENIGN/Tuesday']   = df_R10
particiones['R11 BENIGN/Wednesday'] = df_R11
particiones['R12 BENIGN/Thursday']  = df_R12
particiones['R13 BENIGN/Friday']    = df_R13

# Recalcular tabla completa
print(f'{"Partición":<32} {"N registros":>12} {"P(partición)":>13}')
print('-' * 59)
suma = 0
for nombre, part in particiones.items():
    n = part.count()
    suma += n
    print(f'{nombre:<32} {n:>12,} {n/total:>13.4f}')
print('-' * 59)
print(f'{"TOTAL":<32} {suma:>12,} {suma/total:>13.4f}')

Partición                         N registros  P(partición)
-----------------------------------------------------------
R1 BENIGN/Monday                      529,903        0.1872
R2 BruteForce/Tuesday                  13,835        0.0049
R3 DoS/Wednesday                      252,661        0.0893
R4 Heartbleed/Wednesday                    11        0.0000
R5 WebAttack/Thursday                   2,180        0.0008
R6 Infiltration/Thursday                   36        0.0000
R7 Botnet/Friday                        1,966        0.0007
R8 DDoS/Friday                        128,027        0.0452
R9 PortScan/Friday                    158,930        0.0561
R10 BENIGN/Tuesday                    432,057        0.1526
R11 BENIGN/Wednesday                  440,010        0.1554
R12 BENIGN/Thursday                   456,737        0.1614
R13 BENIGN/Friday                     414,275        0.1464
-----------------------------------------------------------
TOTAL                               2,83

---
## Sección 16: Extracción de Sub-muestras de Prueba

Para verificar el correcto funcionamiento del código de particionamiento, extraemos sub-muestras de prueba de cada partición utilizando el método `sample()` de PySpark. Establecemos un objetivo de aproximadamente 1,000 registros por partición y calculamos la fracción de muestreo necesaria como `TARGET / N_partición`. Para las particiones ultra-minoritarias R4 (Heartbleed, n=11) y R6 (Infiltration, n=36), la fracción calculada supera 1.0, por lo que la limitamos a 1.0 mediante `min()`, lo que equivale a incluir el 100% de sus registros (muestreo censal).

Utilizamos `seed=42` en todas las llamadas a `sample()` para garantizar la reproducibilidad: cualquier ejecución posterior del notebook producirá exactamente las mismas sub-muestras. El parámetro `withReplacement=False` asegura que cada registro puede ser seleccionado como máximo una vez, lo que corresponde al Muestreo Aleatorio Simple sin reemplazo recomendado por la teoría estadística para poblaciones finitas (Ahmed, 2024).

> **Nota:** Estas sub-muestras son únicamente de verificación. En la etapa posterior de aprendizaje automático, este mismo código se usará con fracciones calculadas estadísticamente para generar los conjuntos definitivos de entrenamiento y prueba.

In [ ]:
SEED = 42
TARGET = 1000  # registros de prueba por partición

print(f'{"Partición":<32} {"N partición":>12} {"Fracción":>10} {"N muestra":>10}')
print('-' * 66)

muestras = {}
for nombre, part in particiones.items():
    n = part.count()
    # Si la partición tiene menos registros que TARGET, tomar todo (censo)
    frac = min(TARGET / n, 1.0) if n > 0 else 0
    muestra = part.sample(withReplacement=False, fraction=frac, seed=SEED)
    n_muestra = muestra.count()
    muestras[nombre] = muestra
    print(f'{nombre:<32} {n:>12,} {frac:>10.4f} {n_muestra:>10,}')

Partición                         N partición   Fracción  N muestra
------------------------------------------------------------------
R1 BENIGN/Monday                      529,903     0.0019        955
R2 BruteForce/Tuesday                  13,835     0.0723      1,013
R3 DoS/Wednesday                      252,661     0.0040        996
R4 Heartbleed/Wednesday                    11     1.0000         11
R5 WebAttack/Thursday                   2,180     0.4587      1,025
R6 Infiltration/Thursday                   36     1.0000         36
R7 Botnet/Friday                        1,966     0.5086      1,029
R8 DDoS/Friday                        128,027     0.0078      1,011
R9 PortScan/Friday                    158,930     0.0063        990
R10 BENIGN/Tuesday                    432,057     0.0023      1,040
R11 BENIGN/Wednesday                  440,010     0.0023      1,025
R12 BENIGN/Thursday                   456,737     0.0022      1,031
R13 BENIGN/Friday                     414,275    

---
## Sección 17: Verificación de Distribución de Etiquetas en Sub-muestras de Ataque

Como paso final de validación, revisamos la distribución de etiquetas (`Label`) dentro de las sub-muestras extraídas de las particiones de ataque. El objetivo de esta verificación es confirmar que el muestreo preservó la diversidad de sub-clases dentro de cada partición: por ejemplo, que R3 (DoS) contiene registros de los cuatro tipos de DoS (Hulk, GoldenEye, Slowloris, Slowhttptest) y que R5 (WebAttack) contiene los tres tipos de ataques web (Brute Force, XSS, SQL Injection), incluyendo el ultra-minoritario SQL Injection.

Esta verificación es especialmente importante para las particiones R4 (Heartbleed) y R6 (Infiltration), donde confirmamos que se incluyeron los 11 y 36 registros respectivamente, validando el comportamiento de muestreo censal para clases ultra-minoritarias. Si alguna sub-clase hubiera desaparecido de la muestra, habría sido necesario ajustar la estrategia de muestreo para esa partición específica.

In [ ]:
# Solo revisamos las particiones de ataque para confirmar que los labels son correctos
ataques = ['R2 BruteForce/Tuesday', 'R3 DoS/Wednesday',
           'R4 Heartbleed/Wednesday', 'R5 WebAttack/Thursday',
           'R6 Infiltration/Thursday', 'R7 Botnet/Friday',
           'R8 DDoS/Friday', 'R9 PortScan/Friday']

for nombre in ataques:
    print(f'\n=== Muestra {nombre} ===')
    muestras[nombre].groupBy('Label') \
                    .count() \
                    .orderBy('count', ascending=False) \
                    .show(truncate=False)


=== Muestra R2 BruteForce/Tuesday ===
+-----------+-----+
|Label      |count|
+-----------+-----+
|FTP-Patator|566  |
|SSH-Patator|447  |
+-----------+-----+


=== Muestra R3 DoS/Wednesday ===
+----------------+-----+
|Label           |count|
+----------------+-----+
|DoS Hulk        |913  |
|DoS GoldenEye   |41   |
|DoS slowloris   |22   |
|DoS Slowhttptest|20   |
+----------------+-----+


=== Muestra R4 Heartbleed/Wednesday ===
+----------+-----+
|Label     |count|
+----------+-----+
|Heartbleed|11   |
+----------+-----+


=== Muestra R5 WebAttack/Thursday ===
+-------------------------+-----+
|Label                    |count|
+-------------------------+-----+
|Web Attack  Brute Force  |721  |
|Web Attack  XSS          |295  |
|Web Attack  Sql Injection|9    |
+-------------------------+-----+


=== Muestra R6 Infiltration/Thursday ===
+------------+-----+
|Label       |count|
+------------+-----+
|Infiltration|36   |
+------------+-----+


=== Muestra R7 Botnet/Friday ===
+-----+-